# Support Ticket Classifier (Transformer-Based)
Step-wise implementation with clear explanations.

In [ ]:
# Step 1: Install Libraries
!pip install transformers datasets pandas scikit-learn torch

In [ ]:
# Step 2: Load Dataset
from datasets import load_dataset
import pandas as pd
dataset = load_dataset('Tobi-Bueck/customer-support-tickets')
df = dataset['train'].to_pandas()
df.head()

In [ ]:
# Step 3: Prepare Data
print(df.columns)
df = df[['text','label']]
df = df.dropna()
df.head()

In [ ]:
# Step 4: Encode Labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])
print(dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
# Step 5: Train-Test Split
from sklearn.model_selection import train_test_split
train_texts, val_texts, train_labels, val_labels = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)

In [ ]:
# Step 6: Tokenization
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True)

In [ ]:
# Step 7: Dataset Class
import torch
class TicketDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item
    def __len__(self):
        return len(self.labels)
train_dataset = TicketDataset(train_encodings, train_labels)
val_dataset = TicketDataset(val_encodings, val_labels)

In [ ]:
# Step 8: Load Model
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(le.classes_))

In [ ]:
# Step 9: Training Arguments
from transformers import TrainingArguments
training_args = TrainingArguments(output_dir='./results', num_train_epochs=3, per_device_train_batch_size=8, per_device_eval_batch_size=8, evaluation_strategy='epoch')

In [ ]:
# Step 10: Train Model
from transformers import Trainer
trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=val_dataset)
trainer.train()

In [ ]:
# Step 11: Evaluate
trainer.evaluate()

In [ ]:
# Step 12: Detailed Metrics
from sklearn.metrics import classification_report
predictions = trainer.predict(val_dataset)
preds = predictions.predictions.argmax(axis=1)
print(classification_report(val_labels, preds))

In [ ]:
# Step 13: Prediction Function
def predict_ticket(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class_id = logits.argmax().item()
    return le.inverse_transform([predicted_class_id])[0]
print(predict_ticket('Refund not received'))